In [44]:
import json
from litellm import completion
import litellm
import sys
import os
from datetime import datetime
from typing import List, Optional, Dict, Any
import argparse
from dataclasses import dataclass

# CRITICAL: Completely reset all LiteLLM and API-related environment variables
litellm.custom_provider_map = []
litellm.api_base = None  # Reset any global api_base
litellm.api_key = None   # Reset any global api_key

# Clear all OpenAI-related environment variables that might interfere
for env_var in ['OPENAI_API_BASE', 'OPENAI_BASE_URL', 'OPENAI_API_KEY']:
    if env_var in os.environ:
        del os.environ[env_var]
        print(f"🔧 DEBUG: Cleared environment variable {env_var}")

# Also clear any other API base URLs that might interfere
for env_var in ['API_BASE', 'LITELLM_API_BASE']:
    if env_var in os.environ:
        del os.environ[env_var]
        print(f"🔧 DEBUG: Cleared environment variable {env_var}")

print("🔧 DEBUG: Completely reset LiteLLM configuration and environment variables")
print("🔧 DEBUG: Will configure vLLM per-request to avoid global conflicts")

# Add the tau-bench-lightning directory to path to find tau_bench module
# Use getcwd() instead of __file__ for notebook compatibility
tau_bench_path = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))
if os.path.exists(tau_bench_path) and tau_bench_path not in sys.path:
    sys.path.insert(0, tau_bench_path)

from tau_bench.agents.base import Agent
from tau_bench.agents import debug_message_to_action
from tau_bench.envs.base import Env
from tau_bench.types import SolveResult, Action, RESPOND_ACTION_NAME


def convert_messages_for_gemma(messages: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    """
    Convert OpenAI-style messages to Gemma-compatible format.
    Gemma chat template requires alternating user/assistant roles, so we need to 
    properly format messages to maintain this alternation while preserving content.
    """
    gemma_messages = []
    last_role = None
    
    for msg in messages:
        role = msg.get("role", "user")
        content = msg.get("content", "")
        
        if role == "system":
            # System message becomes a user message with clear system indicator
            new_content = f"# System Instructions\n\n{content}"
            target_role = "system"
            
        elif role == "assistant":
            # Assistant message stays as assistant
            if msg.get("tool_calls"):
                # Assistant made a tool call
                tool_call = msg["tool_calls"][0]  # Take first tool call
                func_name = tool_call["function"]["name"]
                func_args = tool_call["function"]["arguments"]
                new_content = f"# Tool Call\n\nFunction: {func_name}\nArguments: {func_args}"
            else:
                # Regular assistant response
                new_content = content
            target_role = "assistant"
            
        elif role == "tool":
            # Tool result becomes a user message with tool indicator
            tool_name = msg.get("name", "unknown")
            new_content = f"# Tool Result ({tool_name})\n\n{content}"
            target_role = "user"
            
        else:
            # Regular user message
            new_content = content
            target_role = "user"
        
        # Handle role alternation requirement - if same role as last, combine messages
        if last_role == target_role and len(gemma_messages) > 0:
            # Combine with previous message of same role
            gemma_messages[-1]["content"] += f"\n\n---\n\n{new_content}"
        else:
            # Add new message
            gemma_messages.append({"role": target_role, "content": new_content})
            last_role = target_role
    
    # Ensure we start with user message for proper alternation
    if len(gemma_messages) > 0 and gemma_messages[0]["role"] != "user":
        gemma_messages.insert(0, {"role": "user", "content": "# Initial Context\n\nStarting conversation."})
    
    return gemma_messages


class VLLMToolCallingAgent(Agent):
    def __init__(
        self,
        tools_info: Optional[List[Dict[str, Any]]],
        wiki: str,
        model: str,
        provider: str,
        temperature: float = 0.0,
    ):
        # DEBUGGING: Select specific tools for exchange scenario
        relevant_tool_names = [
            'find_user_id_by_name_zip', 
            'get_order_details', 
            'get_product_details',
            'exchange_delivered_order_items'
        ]
        
        # Handle None tools_info
        if tools_info is None:
            self.tools_info = []
        else:
            # self.tools_info = [tool for tool in tools_info if tool['function']['name'] in relevant_tool_names]
            self.tools_info = tools_info
        self.wiki = wiki
        self.model = model
        self.provider = provider
        self.temperature = temperature
        
        print(f"🔧 DEBUG: VLLM Agent initialized with:")
        print(f"   Model: {model}")
        print(f"   Provider: {provider}")
        print(f"   Temperature: {temperature}")
        print(f"   Wiki length: {len(wiki)}")
        print(f"   Tools count: {len(tools_info) if tools_info else 0} -> LIMITED TO: {len(self.tools_info)}")
        if self.tools_info:
            print(f"   Available tools: {[tool['function']['name'] for tool in self.tools_info]}")
        else:
            print(f"   Available tools: None (no tools provided)")
        
        # Handle vLLM provider mapping - always use OpenAI-compatible API
        self.litellm_provider = "openai"
        self.vllm_base_url = "http://127.0.0.1:8001/v1"  # Use 127.0.0.1 as requested
        
        # Set the model name for OpenAI-compatible vLLM server
        if model.startswith("vllm/"):
            model_name = model.replace("vllm/", "")
        else:
            model_name = model
        
        self.litellm_model = model_name
        
        print(f"🔧 DEBUG: Using vLLM via OpenAI-compatible API:")
        print(f"   Model: {self.litellm_model}")
        print(f"   Base URL: {self.vllm_base_url}")
        print(f"   Provider: {self.litellm_provider}")

    def solve(
        self, env: Env, task_index: Optional[int] = None, max_num_steps: int = 30
    ) -> SolveResult:
        print(f"\n🚀 DEBUG: Starting VLLM solve for task {task_index}")
        
        # Create detailed log file in log/ directory - use getcwd() instead of __file__
        import os
        log_dir = os.path.join(os.getcwd(), "..", "log")
        os.makedirs(log_dir, exist_ok=True)
        log_filename = os.path.join(log_dir, f"vllm_gemma3_debug_detailed_task_{task_index}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log")
        
        def log_and_print(message):
            """Helper function to both print and log messages to log/ directory"""
            print(message)
            with open(log_filename, "a", encoding="utf-8") as log_file:
                log_file.write(message + "\n")
        
        log_and_print(f"🚀 DEBUG: Starting VLLM solve for task {task_index}")
        log_and_print(f"📁 DEBUG: Log file: {log_filename}")
        log_and_print(f"📅 Timestamp: {datetime.now().isoformat()}")
        log_and_print(f"🔧 Model: {self.model} via {self.provider} (vLLM)")
        log_and_print("=" * 100)
        
        total_cost = 0.0
        env_reset_res = env.reset(task_index=task_index)
        obs = env_reset_res.observation
        info = env_reset_res.info.model_dump()
        reward = 0.0
        
        log_and_print(f"🔧 DEBUG: Initial observation: {obs}")
        
        # Create standard OpenAI format messages first
        messages: List[Dict[str, Any]] = [
            {"role": "system", "content": self.wiki},
            {"role": "user", "content": obs},
        ]
        
        log_and_print(f"🔧 DEBUG: Initial messages constructed:")
        log_and_print(f"   System message length: {len(messages[0]['content'])}")
        log_and_print(f"   User message: {messages[1]['content']}")
        log_and_print(f"   🎯 Task index being run: {task_index}")
        
        # Print the full system message to understand what instructions are being sent
        log_and_print(f"\n📋 FULL SYSTEM MESSAGE (WIKI):")
        log_and_print("=" * 80)
        log_and_print(messages[0]['content'])
        log_and_print("=" * 80)
        
        log_and_print(f"\n📋 FULL USER INSTRUCTION:")
        log_and_print("=" * 80)
        log_and_print(messages[1]['content'])
        log_and_print("=" * 80)
        
        for step in range(max_num_steps):
            log_and_print(f"\n📍 DEBUG: turn {step + 1}/{max_num_steps}")
            log_and_print("🔥" * 60)
            
            # Show the exact parameters being sent
            log_and_print(f"🔧 DEBUG: LiteLLM vLLM call parameters:")
            log_and_print(f"   model: {self.litellm_model}")
            log_and_print(f"   custom_llm_provider: {self.litellm_provider}")
            log_and_print(f"   api_base: {self.vllm_base_url}")
            log_and_print(f"   temperature: {self.temperature}")
            log_and_print(f"   tools count: {len(self.tools_info)}")
            log_and_print(f"   messages count: {len(messages)}")
            
            # Log the COMPLETE conversation context being sent to the model
            log_and_print(f"\n💬 COMPLETE CONVERSATION CONTEXT SENT TO MODEL:")
            log_and_print("🔸" * 80)
            for i, msg in enumerate(messages):
                log_and_print(f"MESSAGE {i+1} (Role: {msg['role']}):")
                log_and_print(f"Content: {msg.get('content', 'N/A')}")
                if 'tool_call_id' in msg:
                    log_and_print(f"Tool Call ID: {msg['tool_call_id']}")
                if 'name' in msg:
                    log_and_print(f"Tool Name: {msg['name']}")
                log_and_print("-" * 40)
            log_and_print("🔸" * 80)
            
            # Make the LiteLLM call - only pass tools if we have them
            # Convert messages to Gemma-compatible format (only 'user' role supported)
            gemma_messages = convert_messages_for_gemma(messages)
            
            # 🔧 NO TRUNCATION - Using full input prompt size
            total_words = sum(len(msg.get('content', '').split()) for msg in gemma_messages)
            total_chars = sum(len(msg.get('content', '')) for msg in gemma_messages)
            log_and_print(f"🔧 DEBUG: FULL INPUT SIZE - NO TRUNCATION APPLIED")
            log_and_print(f"   Total words: {total_words}")
            log_and_print(f"   Total characters: {total_chars}")
            log_and_print(f"   Estimated tokens: ~{int(total_words / 0.75)}")
            log_and_print(f"   Messages count: {len(gemma_messages)}")
            
            # CRITICAL: Configure vLLM parameters per-request only
            completion_params = {
                "messages": gemma_messages,
                "model": self.litellm_model,
                "temperature": self.temperature,
                "api_base": self.vllm_base_url,
                "api_key": "EMPTY",  # vLLM doesn't require API key
                "custom_llm_provider": "openai",  # Use OpenAI-compatible provider
            }
            
            # Only add tools if we have them - NOTE: Gemma may not support tools properly
            if self.tools_info:
                completion_params["tools"] = self.tools_info
                log_and_print(f"🔧 DEBUG: Including {len(self.tools_info)} tools in request")
            else:
                log_and_print(f"🔧 DEBUG: No tools included in request")
            
            # Log the exact completion parameters
            log_and_print("🟡" * 80)
            log_and_print(json.dumps(completion_params, indent=2, ensure_ascii=False, default=str))
            log_and_print("🟡" * 80)
                
            try:
                log_and_print(f"\n🚀 CALLING vLLM via LiteLLM...")
                
                # Try to extract the actual prompt that will be sent to the model
                try:
                    # Method: Use transformers to apply chat template and see exact prompt
                    log_and_print(f"\n🔍 EXTRACTING EXACT PROMPT VIA TRANSFORMERS...")
                    from transformers import AutoTokenizer
                    
                    # Use the actual model tokenizer to see the exact prompt format
                    if "gemma" in self.litellm_model.lower():
                        tokenizer_id = "google/gemma-3-12bb-it"  # Use 2b as proxy for chat template
                    else:
                        tokenizer_id = self.litellm_model
                    
                    tokenizer = AutoTokenizer.from_pretrained(tokenizer_id, trust_remote_code=True)
                    
                    # Convert our gemma messages back to standard format for tokenizer
                    tokenizer_messages = []
                    for msg in gemma_messages:
                        if msg["content"].startswith("# System Instructions"):
                            content = msg["content"].replace("# System Instructions\n\n", "")
                            tokenizer_messages.append({"role": "system", "content": content})
                        else:
                            tokenizer_messages.append({"role": "user", "content": msg["content"]})
                    
                    if hasattr(tokenizer, 'apply_chat_template'):
                        exact_prompt = tokenizer.apply_chat_template(
                            tokenizer_messages,
                            tokenize=False,
                            add_generation_prompt=True
                        )
                        
                        log_and_print(f"✅ EXACT PROMPT THAT WILL BE SENT TO MODEL:")
                        log_and_print("🟢" * 80)
                        log_and_print(exact_prompt)
                        log_and_print("🟢" * 80)
                        
                        # Also get token count
                        tokens = tokenizer.apply_chat_template(
                            tokenizer_messages,
                            tokenize=True,
                            add_generation_prompt=True
                        )
                        log_and_print(f"🔢 EXACT TOKEN COUNT: {len(tokens)} tokens")
                    else:
                        log_and_print(f"⚠️ Tokenizer doesn't support chat templates")
                        
                except Exception as prompt_e:
                    log_and_print(f"⚠️ Could not extract exact prompt: {prompt_e}")
                
                res = completion(**completion_params)
                log_and_print(f"✅ DEBUG: LiteLLM vLLM response received")
                log_and_print(f"   Model: {res.model}")
                
                # Safe cost extraction
                response_cost = 0.0
                if hasattr(res, '_hidden_params') and res._hidden_params:
                    response_cost = res._hidden_params.get('response_cost', 0.0) or 0.0
                log_and_print(f"   Response cost: {response_cost}")
                
            except Exception as e:
                log_and_print(f"❌ DEBUG: LiteLLM vLLM call failed: {e}")
                import traceback
                log_and_print(f"Full traceback: {traceback.format_exc()}")
                break
            
            # Extract and analyze the message
            next_message = res.choices[0].message.model_dump()
            
            log_and_print(f"\n🤖 DEBUG: Raw message from vLLM model:")
            log_and_print(f"   Role: {next_message.get('role')}")
            log_and_print(f"   Content: {next_message.get('content')}")
            log_and_print(f"   Tool calls: {next_message.get('tool_calls')}")
            
            # Log the COMPLETE raw response
            log_and_print(f"\n📄 COMPLETE RAW vLLM MODEL RESPONSE:")
            log_and_print("🔹" * 80)
            log_and_print(json.dumps(next_message, indent=2, ensure_ascii=False))
            log_and_print("🔹" * 80)
            
            total_cost += response_cost
            
            # Convert to action with debugging
            action = debug_message_to_action(next_message)
            
            log_and_print(f"🎯 DEBUG: Action created:")
            log_and_print(f"   Name: {action.name}")
            log_and_print(f"   Kwargs: {action.kwargs}")
            
            # Execute the action
            env_response = env.step(action)
            reward = env_response.reward
            info = {**info, **env_response.info.model_dump()}
            
            log_and_print(f"🔧 DEBUG: Environment response:")
            log_and_print(f"   Reward: {reward}")
            log_and_print(f"   Done: {env_response.done}")
            log_and_print(f"   Observation: {env_response.observation}")
            
            # Handle conversation continuation
            if action.name != RESPOND_ACTION_NAME:
                log_and_print(f"🔧 DEBUG: Adding tool call to conversation")
                
                # Check if we have actual tool_calls or if we parsed from content
                if next_message.get("tool_calls") is not None and len(next_message["tool_calls"]) > 0:
                    # Standard tool_calls format
                    next_message["tool_calls"] = next_message["tool_calls"][:1]
                    messages.extend(
                        [
                            next_message,
                            {
                                "role": "tool",
                                "tool_call_id": next_message["tool_calls"][0]["id"],
                                "name": next_message["tool_calls"][0]["function"]["name"],
                                "content": env_response.observation,
                            },
                        ]
                    )
                else:
                    # We parsed function call from content - simulate tool_calls structure
                    fake_tool_call = {
                        "id": f"call_{action.name}_{len(messages)}",
                        "type": "function",
                        "function": {
                            "name": action.name,
                            "arguments": json.dumps(action.kwargs)
                        }
                    }
                    next_message["tool_calls"] = [fake_tool_call]
                    messages.extend(
                        [
                            next_message,
                            {
                                "role": "tool",
                                "tool_call_id": fake_tool_call["id"],
                                "name": action.name,
                                "content": env_response.observation,
                            },
                        ]
                    )
                    
            else:
                log_and_print(f"🔧 DEBUG: Adding text response to conversation")
                messages.extend(
                    [
                        next_message,
                        {"role": "user", "content": env_response.observation},
                    ]
                )
            
            # Log the updated conversation state
            log_and_print(f"\n📝 CONVERSATION STATE AFTER TURN {step + 1}:")
            log_and_print(f"   Total messages: {len(messages)}")
            log_and_print(f"   Current reward: {reward}")
            log_and_print(f"   Task done: {env_response.done}")
            
            if env_response.done:
                log_and_print(f"🏁 DEBUG: Task completed at step {step + 1}")
                break
                
        log_and_print(f"🎯 DEBUG: Final result - Reward: {reward}, Total cost: {total_cost}")
        log_and_print(f"💾 DEBUG: Complete log saved to: {log_filename}")
        
        return SolveResult(
            reward=reward,
            info=info,
            messages=messages,
            total_cost=total_cost,
        )

print("✅ VLLMToolCallingAgent class defined successfully!")

🔧 DEBUG: Completely reset LiteLLM configuration and environment variables
🔧 DEBUG: Will configure vLLM per-request to avoid global conflicts
✅ VLLMToolCallingAgent class defined successfully!


In [45]:
import argparse
import json
import os
import random
import time
import multiprocessing
from datetime import datetime
from typing import List
import sys
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

# Add the tau-bench-lightning directory to path to find tau_bench module
# Use getcwd() instead of __file__ for notebook compatibility
tau_bench_path = os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd())))
if os.path.exists(tau_bench_path) and tau_bench_path not in sys.path:
    sys.path.insert(0, tau_bench_path)

# Add sft_training to path for SFT model integration (if needed)
sft_path = os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.getcwd()))), "sft_training")
if os.path.exists(sft_path) and sft_path not in sys.path:
    sys.path.insert(0, sft_path)

from tau_bench.agents.base import Agent
from tau_bench.envs import get_env
from tau_bench.types import EnvRunResult
from tau_bench.run import display_metrics
from litellm import completion

# Configuration setup based on the specified parameters
@dataclass
class Config:
    agent_strategy: str = "tool-calling"
    env: str = "retail"
    model: str = "gemma3:12b"
    model_provider: str = "openai"  # Changed to openai for vLLM compatibility
    user_model: str = "gemini-2.5-pro"
    user_model_provider: str = "vertex_ai"
    user_strategy: str = "llm"
    max_concurrency: int = 1
    task_id: int = 0  # Set to 0 to run task 0 by default
    user_temperature: float = 0.0
    task_split: str = "test"
    end_index: int = -1
    start_index: int = 0
    task_ids: List[int] = None
    num_trials: int = 1
    shuffle: bool = False

def parse_args():
    parser = argparse.ArgumentParser(description="Run vllm gemma3 debug with tau-bench")
    parser.add_argument("--max-concurrency", type=int, default=1,
                        help="Maximum number of concurrent tasks (default: 1)")
    parser.add_argument("--end-index", type=int, default=-1,
                        help="End index for task range, -1 for all tasks (default: -1)")
    parser.add_argument("--start-index", type=int, default=0,
                        help="Start index for task range (default: 0)")
    parser.add_argument("--task-id", type=int, default=0,
                        help="Single task ID to run (overrides start/end index)")
    parser.add_argument("--model", type=str, default="gemma3:12b",
                        help="Model to use (default: gemma3:12b)")
    parser.add_argument("--model-provider", type=str, default="openai",
                        help="Model provider (default: openai for vLLM)")
    parser.add_argument("--user-model", type=str, default="gemini-2.5-pro",
                        help="User model (default: gemini-2.5-pro)")
    parser.add_argument("--user-model-provider", type=str, default="vertex_ai",
                        help="User model provider (default: vertex_ai)")
    parser.add_argument("--num-trials", type=int, default=1,
                        help="Number of trials per task (default: 1)")
    parser.add_argument("--shuffle", action="store_true",
                        help="Shuffle task order")
    return parser.parse_args()

# Check if we're running in a notebook environment
def is_notebook():
    try:
        # Check if we're in IPython/Jupyter
        from IPython import get_ipython
        if get_ipython() is not None:
            return True
    except ImportError:
        pass
    
    # Check if we're being run by a kernel
    if any('ipykernel_launcher' in arg for arg in sys.argv):
        return True
    
    return False

# Handle notebook vs script execution
config = Config()  # Initialize with defaults first

if not is_notebook():
    # Script mode - parse arguments normally
    try:
        args = parse_args()
        config.max_concurrency = args.max_concurrency
        config.end_index = args.end_index
        config.start_index = args.start_index
        config.task_id = 0  # Force task_id to 0 for debugging
        config.model = args.model
        config.model_provider = args.model_provider
        config.user_model = args.user_model
        config.user_model_provider = args.user_model_provider
        config.num_trials = args.num_trials
        config.shuffle = args.shuffle
        print("🔧 DEBUG: Running in script mode with parsed arguments")
    except SystemExit as e:
        print(f"🔧 DEBUG: Argument parsing failed: {e}")
        print("🔧 DEBUG: Using default configuration")
else:
    # Notebook mode - use defaults
    print("🔧 DEBUG: Running in notebook mode with default configuration")

# Set up task_ids to specifically run task 0
config.task_ids = [0]  # Force to run only task 0

print(f"🔧 DEBUG: Configuration setup (FORCED TO TASK 0):")
print(f"   task_id: {config.task_id}")
print(f"   task_ids: {config.task_ids}")
print(f"   start_index: {config.start_index}")
print(f"   end_index: {config.end_index}")
print(f"   max_concurrency: {config.max_concurrency}")
print(f"   env: {config.env}")
print(f"   task_split: {config.task_split}")
print(f"   model: {config.model}")
print(f"   model_provider: {config.model_provider}")
print(f"   num_trials: {config.num_trials}")
print(f"   shuffle: {config.shuffle}")

🔧 DEBUG: Running in notebook mode with default configuration
🔧 DEBUG: Configuration setup (FORCED TO TASK 0):
   task_id: 0
   task_ids: [0]
   start_index: 0
   end_index: -1
   max_concurrency: 1
   env: retail
   task_split: test
   model: gemma3:12b
   model_provider: openai
   num_trials: 1
   shuffle: False


In [46]:

# Initialize environment
env = get_env(
    config.env,
    user_strategy=config.user_strategy,
    user_model=config.user_model,
    user_provider=config.user_model_provider,
    user_temperature=config.user_temperature,
    task_split=config.task_split,
)


In [47]:

# Agent factory function
def agent_factory(tools_info, wiki, config):
    return VLLMToolCallingAgent(
        tools_info=tools_info,
        wiki=wiki,
        model=config.model,
        provider=config.model_provider,
        temperature=0.0
    )


In [48]:

# Checkpoint path setup - save to log/ directory using getcwd()
log_dir = os.path.join(os.getcwd(), "log")
os.makedirs(log_dir, exist_ok=True)
ckpt_path = os.path.join(log_dir, f"vllm_gemma3_debug_results_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")

# Start timing
start_time = time.time()

# Read retail_tools_info.md and append to env.wiki
retail_tools_path = "/home/lixali_google_com/data/adk-lightning/tau-bench-lightning/retail_tools_info.md"
try:
    with open(retail_tools_path, "r") as f:
        retail_tools_content = f.read()
    enhanced_wiki = env.wiki + "\n\n" + "="*80 + "\n" + "AVAILABLE TOOLS DOCUMENTATION:\n" + "="*80 + "\n\n" + retail_tools_content
    print(f"🔧 DEBUG: Enhanced wiki with retail tools info ({len(retail_tools_content)} chars added)")
except FileNotFoundError:
    print(f"⚠️  WARNING: retail_tools_info.md not found at {retail_tools_path}, using original wiki")
    enhanced_wiki = env.wiki

agent = agent_factory(
    wiki=enhanced_wiki,
    config=config,
    tools_info=None,  # Set to None to disable tools
)

end_index = (
    len(env.tasks) if config.end_index == -1 else min(config.end_index, len(env.tasks))
)
results: List[EnvRunResult] = []

if config.task_ids and len(config.task_ids) > 0:
    print(f"Running tasks {config.task_ids} (checkpoint path: {ckpt_path})")
else:
    print(f"Running tasks {config.start_index} to {end_index} (checkpoint path: {ckpt_path})")

for i in range(config.num_trials):
    if config.task_ids and len(config.task_ids) > 0:
        idxs = config.task_ids
    else:
        idxs = list(range(config.start_index, end_index))
    if config.shuffle:
        random.shuffle(idxs)

    for idx in idxs:
        isolated_env = get_env(
            config.env,
            user_strategy=config.user_strategy,
            user_model=config.user_model,
            task_split=config.task_split,
            user_provider=config.user_model_provider,
            user_temperature=config.user_temperature,
            task_index=idx,
        )

        print(f"Running task {idx}")
        try:
            res = agent.solve(
                env=isolated_env,
                task_index=idx,
            )
            result = EnvRunResult(
                task_id=idx,
                reward=res.reward,
                info=res.info,
                traj=res.messages,
                trial=i,
            )
        except Exception as e:
            result = EnvRunResult(
                task_id=idx,
                reward=0.0,
                info={"error": str(e)},
                traj=[],
                trial=i,
            )
        print("✅" if result.reward == 1 else "❌", f"task_id={idx}", result.info)
        print("-----")
        results.append(result)
        with open(ckpt_path, "a") as f:
            json.dump(result.model_dump(), f, indent=2)
            f.write("\n")

metrics = display_metrics(results)

# Calculate total runtime
end_time = time.time()
total_runtime = end_time - start_time
runtime_minutes = total_runtime / 60
runtime_hours = total_runtime / 3600

# Print runtime to terminal
print(f"\n⏱️  Total Runtime: {total_runtime:.2f} seconds ({runtime_minutes:.2f} minutes, {runtime_hours:.2f} hours)")

🔧 DEBUG: Enhanced wiki with retail tools info (20154 chars added)
🔧 DEBUG: VLLM Agent initialized with:
   Model: gemma3:12b
   Provider: openai
   Temperature: 0.0
   Wiki length: 26069
   Tools count: 0 -> LIMITED TO: 0
   Available tools: None (no tools provided)
🔧 DEBUG: Using vLLM via OpenAI-compatible API:
   Model: gemma3:12b
   Base URL: http://127.0.0.1:8001/v1
   Provider: openai
Running tasks [0] (checkpoint path: /home/lixali_google_com/data/adk-lightning/tau-bench-lightning/vllm-gemma3-debug/src/log/vllm_gemma3_debug_results_20250918_225502.json)
Running task 0

🚀 DEBUG: Starting VLLM solve for task 0
🚀 DEBUG: Starting VLLM solve for task 0
📁 DEBUG: Log file: /home/lixali_google_com/data/adk-lightning/tau-bench-lightning/vllm-gemma3-debug/src/../log/vllm_gemma3_debug_detailed_task_0_20250918_225508.log
📅 Timestamp: 2025-09-18T22:55:08.908368
🔧 Model: gemma3:12b via openai (vLLM)
🔧 DEBUG: Initial observation: Hello, I'd like to arrange an exchange for a couple of items from